# MineAgent — Full Training Notebook
**RTX T4 (Kaggle Free)** | Llama 3.2:3b | QLoRA | Stage 1 + Stage 2

## Setup:
1. Add your `session_log.jsonl` as a Kaggle Input Dataset named `mineagent-sessions`
2. Add your HuggingFace token in Kaggle Secrets as `HF_TOKEN`
3. Enable GPU T4 x2 (or T4 single)
4. Run All Cells


In [ ]:
# ── Cell 1: Install ───────────────────────────────────────────────
import subprocess
subprocess.run(['pip','install','-q','unsloth','trl==0.11.0','peft==0.12.0',
    'bitsandbytes','datasets','accelerate','huggingface_hub',
    'minedojo','mwclient','requests'], check=False)
print('Install done')

In [ ]:
# ── Cell 2: GPU + HF Login ────────────────────────────────────────
import torch
from kaggle_secrets import UserSecretsClient
from huggingface_hub import login

print('GPU:', torch.cuda.get_device_name(0))
print('VRAM:', round(torch.cuda.get_device_properties(0).total_memory/1024**3,1),'GB')
print('BF16:', torch.cuda.is_bf16_supported())

HF_TOKEN = UserSecretsClient().get_secret('HF_TOKEN')
login(HF_TOKEN)
print('HuggingFace: logged in')

In [ ]:
# ── Cell 3: Collect Data ─────────────────────────────────────────
import json, time, os, random, requests
from pathlib import Path

Path('/kaggle/working/data').mkdir(exist_ok=True)

SYS_KNOW = 'You are MineAgent, an expert Minecraft AI. Answer accurately about Minecraft.'
SYS_ACT  = 'You are MineAgent. Output ONLY valid JSON: {"action":"NAME","params":{},"reasoning":""}'

def qa_chat(q, a, sys=None):
    return {'messages':[{'role':'system','content':sys or SYS_KNOW},
                        {'role':'user','content':q},
                        {'role':'assistant','content':a}]}

# --- Minecraft Wiki ---
def get_wiki(max_pages=400):
    API='https://minecraft.wiki/api.php'
    CATS=['Blocks','Items','Crafting','Mobs','Biomes','Food','Tools','Weapons','Armor','Mechanics','Brewing','Enchanting']
    pairs, seen = [], set()
    for cat in CATS:
        if len(seen)>=max_pages: break
        r=requests.get(API,params={'action':'query','list':'categorymembers','cmtitle':f'Category:{cat}','cmlimit':'50','format':'json'},timeout=10)
        for p in r.json()['query']['categorymembers']:
            if p['title'] in seen or len(seen)>=max_pages: continue
            seen.add(p['title'])
            r2=requests.get(API,params={'action':'query','titles':p['title'],'prop':'extracts','explaintext':True,'format':'json'},timeout=10)
            txt=list(r2.json()['query']['pages'].values())[0].get('extract','')[:800]
            if len(txt)<80: continue
            pairs.append(qa_chat(f"What is {p['title']} in Minecraft?", txt))
            pairs.append(qa_chat(f"How do you use {p['title']} in Minecraft?", txt[:400]))
            time.sleep(0.25)
    print(f'Wiki: {len(pairs)} pairs from {len(seen)} pages')
    return pairs

# --- MineDojo Reddit ---
def get_reddit(limit=15000):
    try:
        from minedojo.data import RedditDataset
        ds=RedditDataset(full=False, download=True)
    except Exception as e:
        print(f'Reddit skip: {e}'); return []
    KW=['craft','mine','build','survive','wood','stone','coal','iron','diamond','pickaxe','hunger','recipe']
    pairs=[]
    for item in ds:
        if len(pairs)>=limit: break
        t=item.get('title',''); b=item.get('selftext','') or item.get('body','')
        if len(b)<40 or not any(k in (t+b).lower() for k in KW): continue
        pairs.append(qa_chat(t, b[:500]))
    print(f'Reddit: {len(pairs)} pairs'); return pairs

# --- MineDojo YouTube ---
def get_youtube(limit=10000):
    try:
        from minedojo.data import YouTubeDataset
        ds=YouTubeDataset(full=False, download=True)
    except Exception as e:
        print(f'YouTube skip: {e}'); return []
    KW=['wood','mine','craft','survive','build','explore','cave','diamond']
    pairs=[]
    for item in ds:
        if len(pairs)>=limit: break
        t=item.get('title',''); tr=item.get('transcript','')
        if len(tr)<100 or not any(k in t.lower() for k in KW): continue
        ws=tr.split()
        for i in range(0,min(len(ws),600),150):
            chunk=' '.join(ws[i:i+150])
            if len(chunk)>60: pairs.append(qa_chat(f'Minecraft gameplay: {t}', chunk))
    print(f'YouTube: {len(pairs)} pairs'); return pairs

# --- Bot sessions (uploaded by user) ---
def get_sessions():
    path='/kaggle/input/mineagent-sessions/session_log.jsonl'
    if not Path(path).exists(): print('No sessions'); return []
    VALID={'SEEK','MINE','MOVE','CRAFT','EAT','CHAT','FOLLOW','GOTO','STOP'}
    pairs=[]
    for line in open(path):
        item=json.loads(line)
        if item.get('source')!='llm': continue
        a=item.get('action',{})
        if a.get('action') not in VALID: continue
        q=(f"Goal:{item.get('goal','')} "
           f"HP:{item.get('player',{}).get('health',20)} "
           f"Inv:{list(item.get('inventory',{}).keys())[:4]} "
           f"Nearby:{[b['name'] for b in item.get('nearby_blocks',[])[:4]]} "
           f"Progress:{item.get('goal_progress','')}")
        out=json.dumps({'action':a['action'],'params':a.get('params',{}),'reasoning':a.get('reasoning','')})
        pairs.append(qa_chat(q, out, sys=SYS_ACT))
    print(f'Sessions: {len(pairs)} pairs'); return pairs

# Build
stage1 = get_wiki() + get_reddit() + get_youtube()
stage2 = get_sessions()
random.shuffle(stage1); random.shuffle(stage2)

def save(path, data):
    with open(path,'w') as f:
        [f.write(json.dumps(x)+'\n') for x in data]
    print(f'Saved {len(data)} → {path}')

save('/kaggle/working/data/stage1.jsonl', stage1)
save('/kaggle/working/data/stage2.jsonl', stage2)
print(f'Total: {len(stage1)+len(stage2)} examples')

In [ ]:
# ── Cell 4: Train Stage 1 (Knowledge) ────────────────────────────
from unsloth import FastLanguageModel
from trl import SFTTrainer
from transformers import TrainingArguments
from datasets import load_dataset

MODEL_ID = 'unsloth/Llama-3.2-3B-Instruct'

model, tok = FastLanguageModel.from_pretrained(
    model_name=MODEL_ID, max_seq_length=512,
    dtype=torch.bfloat16, load_in_4bit=True)

model = FastLanguageModel.get_peft_model(
    model, r=16, lora_alpha=32,
    target_modules=['q_proj','k_proj','v_proj','o_proj'],
    lora_dropout=0.05, bias='none')

tok.pad_token = tok.eos_token
tok.padding_side = 'right'

ds1 = load_dataset('json', data_files={'train':'/kaggle/working/data/stage1.jsonl'}, split='train')
split1 = ds1.train_test_split(test_size=0.05, seed=42)

def fmt(ex): return {'text': tok.apply_chat_template(ex['messages'], tokenize=False, add_generation_prompt=False)}
split1 = split1.map(fmt)

trainer1 = SFTTrainer(
    model=model, tokenizer=tok,
    train_dataset=split1['train'], eval_dataset=split1['test'],
    dataset_text_field='text', max_seq_length=512,
    args=TrainingArguments(
        output_dir='/kaggle/working/stage1',
        num_train_epochs=3,
        per_device_train_batch_size=4,
        gradient_accumulation_steps=2,
        learning_rate=3e-4,
        lr_scheduler_type='cosine',
        warmup_ratio=0.05,
        bf16=True, logging_steps=50,
        eval_strategy='steps', eval_steps=300,
        save_steps=500, save_total_limit=1,
        report_to='none',
    ))

print('=== Stage 1 Training Start ===')
trainer1.train()
model.save_pretrained('/kaggle/working/stage1-lora')
tok.save_pretrained('/kaggle/working/stage1-lora')
print('Stage 1 complete. Loss:', trainer1.state.log_history[-1])

In [ ]:
# ── Cell 5: Train Stage 2 (Behavior Cloning) ─────────────────────
import gc
del trainer1; gc.collect(); torch.cuda.empty_cache()

model2, tok2 = FastLanguageModel.from_pretrained(
    model_name='/kaggle/working/stage1-lora',
    max_seq_length=1024, dtype=torch.bfloat16, load_in_4bit=True)

model2 = FastLanguageModel.get_peft_model(
    model2, r=16, lora_alpha=32,
    target_modules=['q_proj','k_proj','v_proj','o_proj','gate_proj','up_proj','down_proj'],
    lora_dropout=0.05, bias='none')

tok2.pad_token = tok2.eos_token; tok2.padding_side = 'right'

# ── Load Stage 2 data ────────────────────────────────────────────
# Priority: 1) Bot sessions  2) Synthetic dataset (1500 unique examples)
stage2_path = '/kaggle/working/data/stage2.jsonl'
synth_path  = '/kaggle/input/mineagent-sessions/synthetic_stage2.jsonl'

stage2_size = Path(stage2_path).stat().st_size if Path(stage2_path).exists() else 0

if stage2_size < 500:  # no real session data
    if Path(synth_path).exists():
        import shutil
        shutil.copy(synth_path, stage2_path)
        lines = open(stage2_path).readlines()
        print(f'Loaded {len(lines)} synthetic examples (1500 unique, diverse scenarios)')
    else:
        print('ERROR: synthetic_stage2.jsonl not found!')
        print('Upload training/synthetic_stage2.jsonl to your Kaggle dataset named mineagent-sessions')
        raise FileNotFoundError('Missing synthetic_stage2.jsonl')
else:
    # Merge real sessions + synthetic for best results
    if Path(synth_path).exists():
        with open(stage2_path, 'a') as dst, open(synth_path) as src:
            dst.write(src.read())
        print('Merged real sessions + 1500 synthetic examples')
    lines = open(stage2_path).readlines()
    print(f'Total stage2 examples: {len(lines)}')

ds2 = load_dataset('json', data_files={'train': stage2_path}, split='train')
split2 = ds2.train_test_split(test_size=0.08, seed=42)
split2 = split2.map(lambda x: {'text': tok2.apply_chat_template(
    x['messages'], tokenize=False, add_generation_prompt=False)})

print(f'Train: {len(split2["train"])} | Val: {len(split2["test"])}')

trainer2 = SFTTrainer(
    model=model2, tokenizer=tok2,
    train_dataset=split2['train'], eval_dataset=split2['test'],
    dataset_text_field='text', max_seq_length=1024,
    args=TrainingArguments(
        output_dir='/kaggle/working/stage2',
        num_train_epochs=5,
        per_device_train_batch_size=2,
        gradient_accumulation_steps=4,
        learning_rate=1e-4,
        lr_scheduler_type='cosine',
        warmup_ratio=0.1,
        bf16=True, logging_steps=20,
        eval_strategy='steps', eval_steps=100,
        save_steps=200, save_total_limit=1,
        report_to='none',
    ))

print('=== Stage 2 Training Start ===')
trainer2.train()
model2.save_pretrained('/kaggle/working/mineagent-v1-lora')
tok2.save_pretrained('/kaggle/working/mineagent-v1-lora')
print('Stage 2 done. Loss:', trainer2.state.log_history[-1])

In [ ]:
# ── Cell 6: Export to GGUF ────────────────────────────────────────
import gc
del trainer2, model; gc.collect(); torch.cuda.empty_cache()

print('Converting to GGUF Q4_K_M...')
model2.save_pretrained_gguf(
    '/kaggle/working/mineagent-v1-gguf',
    tok2,
    quantization_method='q4_k_m'
)
import os
files = os.listdir('/kaggle/working/mineagent-v1-gguf')
print('GGUF files:', files)

In [ ]:
# ── Cell 7: Push to HuggingFace Hub ──────────────────────────────
# Model will be at: https://huggingface.co/YOUR_USERNAME/mineagent-v1
from huggingface_hub import HfApi
import subprocess

HF_USERNAME = 'YOUR_HF_USERNAME'  # ← CHANGE THIS
REPO_ID = f'{HF_USERNAME}/mineagent-v1'
api = HfApi()

# Create repo
api.create_repo(repo_id=REPO_ID, repo_type='model', exist_ok=True)

# Upload GGUF
api.upload_folder(
    folder_path='/kaggle/working/mineagent-v1-gguf',
    repo_id=REPO_ID,
    repo_type='model',
)

# Upload LoRA adapter too
api.upload_folder(
    folder_path='/kaggle/working/mineagent-v1-lora',
    repo_id=f'{HF_USERNAME}/mineagent-v1-lora',
    repo_type='model',
)

print(f'Model uploaded!')
print(f'GGUF: https://huggingface.co/{REPO_ID}')
print(f'LoRA: https://huggingface.co/{HF_USERNAME}/mineagent-v1-lora')

## ✅ Training Complete!
Your model is now on HuggingFace. Run `download_model.py` locally to pull it.